# Evaluación experimental de las estructuras

Este notebook compara las cinco familias avanzadas con sus referencias exactas o más simples. Presenta población efectiva, repeticiones, tiempo, memoria y calidad. Lee mediciones incluidas en el repositorio; ejecutar todas sus celdas no lanza experimentos largos.

**Uso:** selecciona el kernel de `.venv` y ejecuta en orden. Cambia la familia y la métrica con los selectores para explorar las comparaciones.

In [ ]:
from pathlib import Path
import sys
import json

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from src.notebook_views import FAMILIES, population_table, benchmark_table, quality_table, plot_comparison, plot_scaling

RUTA = ROOT / 'output/benchmarks/report.json'
informe = json.loads(RUTA.read_text(encoding='utf-8'))
dataset = informe['datasets'][0]
display(pd.DataFrame([{'Plataforma': informe['platform'], 'Python': informe['python'], 'Semilla': informe['seed'], 'Fecha': informe['generated_at']}]))
display(population_table(informe))

## 1. Método y alcance

La evaluación de 1K contiene 100 repeticiones por algoritmo. Las entradas se preparan fuera de la región medida y ambas alternativas usan la misma población. `tracemalloc` registra asignaciones Python actuales y pico; RSS antes/después describe el proceso y no su pico nativo.

Las barras muestran media y desviación estándar. El tiempo total incluye la carga experimental de cada algoritmo; para ANN se separan construcción y consultas más adelante. Los conjuntos experimentales proceden de sesiones de interacción. El grafo experimental usa órdenes simuladas de tres productos consecutivos.

In [ ]:
tabla = benchmark_table(dataset)
display(tabla.round(3))

## 2. Comparar alternativas

Bloom se compara con set; CMS con Counter; MinHash+LSH con Jaccard exhaustivo; HNSW con KNN exacto; PageRank con grado; Louvain con propagación de etiquetas. La primera figura conserva una comparación estática; los selectores permiten explorar las demás. Una menor memoria o tiempo debe interpretarse junto con la calidad.

In [ ]:
fig = plot_comparison(dataset, 'Vecinos vectoriales')
display(fig)
plt.close(fig)

def comparar(familia, metrica):
    fig = plot_comparison(dataset, familia, metrica)
    display(fig)
    plt.close(fig)

display(widgets.interactive(comparar, familia=widgets.Dropdown(options=list(FAMILIES), description='Familia:'), metrica=widgets.Dropdown(options=[('Tiempo total', 'seconds'), ('Memoria pico', 'peak_bytes')], description='Métrica:')))

## 3. Calidad y coste de búsqueda

FPR expresa falsos positivos de Bloom; MAE, el error medio de conteo de CMS; recall y precisión, la recuperación de conjuntos; recall@5, la coincidencia ANN con vecinos exactos. Modularity evalúa las particiones y ARI mide su estabilidad respecto de la primera repetición. Estas métricas describen esta población y consultas, no garantías para cualquier dato.

La segunda tabla separa el coste de construir HNSW del tiempo de todas las consultas registradas. Permite valorar si amortizar el índice resulta útil para el volumen de consultas esperado.

In [ ]:
display(quality_table(dataset).round(6))
filas = []
for algoritmo in ('hnsw', 'knn_exact'):
    metricas = dataset['summary'][algoritmo]
    for clave, nombre in [('build_seconds', 'Construcción'), ('query_seconds', 'Consultas')]:
        if clave in metricas:
            filas.append({'Algoritmo': algoritmo, 'Operación': nombre, 'Media (ms)': metricas[clave]['mean'] * 1000, 'Desviación (ms)': metricas[clave]['stdev'] * 1000, 'Consultas por repetición': dataset['queries']})
display(pd.DataFrame(filas).round(3))

## 4. Escalabilidad y población efectiva

Los cortes de 10K, 100K, 1M y 10M incluidos tienen una repetición por escala. Bloom, CMS y sus controles reciben todos los eventos; vectores y conjuntos están acotados a 300, y el grafo utiliza 900 eventos. La tabla muestra qué crece realmente.

El panel de eventos usa ejes log-log. Si el número de vectores permanece constante, sus puntos se muestran sin conectarlos: no constituyen una curva de crecimiento ANN. Esta exploración no reemplaza una evaluación estadística completa en las cinco escalas.

In [ ]:
ruta_escalas = ROOT / 'output/escalabilidad/report.json'
escalas = json.loads(ruta_escalas.read_text(encoding='utf-8'))
display(population_table(escalas))
fig = plot_scaling(escalas)
display(fig)
plt.close(fig)

## 5. Interpretación

- En poblaciones pequeñas, el método exacto puede resultar más económico que construir un índice aproximado. HNSW debe evaluarse separando construcción y consultas.
- Bloom reduce memoria a cambio de falsos positivos; CMS puede sobreestimar. Las decisiones críticas del pipeline usan confirmación exacta, cuyo coste también debe considerarse.
- Mayor modularidad no implica mayor estabilidad: ambas métricas deben examinarse en el mismo grafo.
- Una sola repetición no permite estimar variabilidad; una desviación cero en ese caso no significa estabilidad demostrada.

## Reproducir los experimentos

Desde la raíz, ejecuta `python benchmarks.py --sizes 1K --repetitions 100 --max-entities 1000 --queries 5`. Para estudiar las cinco escalas, utiliza `--sizes 1K 10K 100K 1M 10M`; el script usa 100 repeticiones salvo 10M, que utiliza cinco por defecto. `--max-entities 0` elimina el recorte de entidades y requiere valorar los recursos disponibles. Conserva los resultados de nuevos experimentos con `--output output/nuevo_experimento`.

Consulta [la guía técnica y las referencias](../docs/GUIA_TECNICA.md) para interpretar ventanas, parámetros, complejidades y límites.